# Mervin/Mervis -- gpt-oss-20b LoRA fine-tune (Google Colab, A100)

LoRA fine-tunes OpenAI's **gpt-oss-20b** (~21B-param MoE) on the Mervin/Mervis
persona dataset with [Unsloth](https://github.com/unslothai/unsloth), exports a
GGUF, and uploads it to Hugging Face -- where `serve.py` auto-downloads it like
the other models.

**Unlike the other arena models (fine-tuned on AWS SageMaker), this one trains
entirely on Google Colab.**

This notebook reflects a run that actually completed on an A100-80GB:
- training: 3 epochs, loss 6.1 -> ~0.5, ~12 min
- export: gpt-oss is natively **MXFP4** (a 4-bit format), so the GGUF comes out
  as `*.MXFP4.gguf` (~13.8 GB) -- llama.cpp keeps the experts in MXFP4 rather
  than re-quantizing to Q4_K_M.

### Before you run
- Runtime -> Change runtime type -> **A100 GPU**.
- Add a Colab **secret** named `HF_TOKEN` (key icon, left sidebar) holding a
  Hugging Face **write** token -- used by the upload cell, never written into the
  notebook.

In [ ]:
# Confirm an A100 (or any CUDA GPU) is attached.
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "|", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

In [ ]:
# Install Unsloth and a dependency set it actually supports. gpt-oss needs
# transformers>=4.55, but Unsloth excludes 4.55.0/4.55.1 and caps the upper
# bound -- do NOT let pip pull the latest transformers/trl/datasets.
!pip install --upgrade -q unsloth unsloth_zoo
!pip install -q "transformers==4.56.2" "datasets>=3.4.1,<4.4.0" "trl>=0.18.2,<=0.24.0,!=0.19.0"
# Colab's torch is bleeding-edge; the preinstalled torchao is built against a
# different ABI and crashes on import. transformers auto-imports torchao when
# present, which would kill the whole import chain. We use bnb 4-bit (not MXFP4
# kernels) for training, so torchao is not needed -- remove it.
!pip uninstall -q -y torchao
import transformers, trl, datasets
print("transformers", transformers.__version__, "| trl", trl.__version__, "| datasets", datasets.__version__)

In [ ]:
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 1024  # persona replies are short

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/gpt-oss-20b",
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,    # auto -> bf16 on A100
    load_in_4bit    = True,    # 4-bit base via bitsandbytes
    full_finetuning = False,
)
print("Loaded gpt-oss-20b in 4-bit")

In [ ]:
# LoRA on attention + MLP projections; Unsloth maps these onto the MoE experts.
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = 32,
    lora_dropout               = 0.05,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 3407,
)
model.print_trainable_parameters()

## Dataset

Read straight from this repo's raw CSV (`prompt,response`, ~262 pairs). The
responses already carry the `<Mervin>...</Mervin><Mervis>...</Mervis>` tags, so
we wrap each row as system/user/assistant and let the gpt-oss chat template
render it at **low reasoning effort** (direct reply, no chain-of-thought).

In [ ]:
import csv, io, urllib.request
from datasets import Dataset

CSV_URL = "https://raw.githubusercontent.com/CarlFreeAiEngineer/merv/main/mervin_mervis_finetune.csv"

SYSTEM_PROMPT = (
    "You are a dual-personality assistant. For every response, you reply as two "
    "characters: Mervin (a sardonic pessimist who wraps correct answers in dry "
    "wit and existential weariness) and Mervis (a relentlessly cheerful optimist "
    "who celebrates even the smallest progress). Format your response with "
    "<Mervin>...</Mervin> followed by <Mervis>...</Mervis>."
)

raw  = urllib.request.urlopen(CSV_URL).read().decode("utf-8")
rows = list(csv.DictReader(io.StringIO(raw)))
print(f"Loaded {len(rows)} rows")

def to_messages(r):
    return {"messages": [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": r["prompt"]},
        {"role": "assistant", "content": r["response"]},
    ]}

def apply_template(messages, add_generation_prompt):
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False,
            add_generation_prompt=add_generation_prompt,
            reasoning_effort="low",
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=add_generation_prompt)

ds = Dataset.from_list([to_messages(r) for r in rows])
ds = ds.map(lambda ex: {"text": apply_template(ex["messages"], False)},
            remove_columns=ds.column_names)
print(ds[0]["text"][:700])

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LEN,
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,   # effective batch 16
        num_train_epochs            = 3,
        learning_rate               = 2e-4,
        warmup_ratio                = 0.1,
        lr_scheduler_type           = "cosine",
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        logging_steps               = 5,
        seed                        = 3407,
        output_dir                  = "outputs",
        report_to                   = "none",
    ),
)
trainer.train()

## Quick sanity check

In [ ]:
FastLanguageModel.for_inference(model)

def ask(q):
    msgs = [{"role":"system","content":SYSTEM_PROMPT}, {"role":"user","content":q}]
    prompt = apply_template(msgs, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, top_p=0.9)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(ask("What is 2+2?"))

## Export GGUF

gpt-oss is natively **MXFP4**; Unsloth detects this and emits a single
`*.MXFP4.gguf` (~13.8 GB), skipping the Q4_K_M pass (the experts stay MXFP4, so
re-quantizing barely changes the size). That file is the deployment target.

In [ ]:
import glob, os, time
t0 = time.time()
model.save_pretrained_gguf("gpt-oss-merv-gguf", tokenizer, quantization_method="q4_k_m")
print("export done in", round(time.time()-t0), "s")
for f in glob.glob("/content/**/*.gguf", recursive=True):
    print(round(os.path.getsize(f)/1e9, 2), "GB ", f)

## Upload to Hugging Face

Pushes the GGUF to `freeideas/merv-gpt-oss-20b` as `model-mxfp4.gguf`, where
`serve.py` auto-downloads it. The token comes from the Colab `HF_TOKEN`
secret -- it is never written into the notebook.

In [ ]:
import glob
from google.colab import userdata
from huggingface_hub import HfApi

tok  = userdata.get("HF_TOKEN")
repo = "freeideas/merv-gpt-oss-20b"
src  = glob.glob("/content/**/*.MXFP4.gguf", recursive=True)[0]

api = HfApi()
print("uploading as model-mxfp4.gguf ->", repo, "(", who := api.whoami(token=tok)["name"], ")")
api.create_repo(repo, repo_type="model", exist_ok=True, token=tok)
api.upload_file(path_or_fileobj=src, path_in_repo="model-mxfp4.gguf",
                repo_id=repo, repo_type="model", token=tok)
print("done -> https://huggingface.co/" + repo)

## In the arena

`serve.py` and `index.html` already carry a `gptoss` entry, so any host running
`serve.py` auto-downloads `model-mxfp4.gguf` from `freeideas/merv-gpt-oss-20b`
on startup and the model shows up in the dropdown. See this folder's
`README.md`.

**Size note:** ~13.8 GB in RAM. Comfortable at 24 GB; tight on a 16 GB CPU box
(keep the context window small).